In [ ]:
from __future__ import annotations
import json
import re
from pathlib import Path
import pandas as pd
import argparse
import pathlib
import os

import numpy as np

import matplotlib.pyplot as plt

def set_fontsize(base_fontsize=15):
    fontsize = base_fontsize
    plt.rcParams.update({
        'font.size': fontsize,
        'axes.titlesize': fontsize * 1,
        'axes.labelsize': fontsize,
        'xtick.labelsize': fontsize * 0.8,
        'ytick.labelsize': fontsize * 0.8,
        'legend.fontsize': fontsize * 0.8,
        'font.family': "Arial"
    })

plt.style.use('default')

set_fontsize()

In [ ]:
# town = "Ludwigshafen"
phase_cut_date = "2023-06-01"
prev_phase_cut_date = "2023-06-01"
cutoff_value = 0.05 # cutoff value for ensemble member selection (fraction of best models)

## Plot predictions for prev for different objectives

In [ ]:
objectives = ["cases_and_conc", "prev_and_conc", "three_objectives"]
towns = ["Koblenz", "Kaiserslautern", "Mainz", "Ludwigshafen", "Trier"]

descriptions = ["Reported cases and\n wastewater data", "Cohort testing and\n wastewater data", "All three\navailable datasets"]

In [ ]:
# load pred
prev_data = {}
for town in towns:
    prev_data[town] = {}
    for objective in objectives:
        prev_data[town][objective] = np.load(f"/home/iru-mls/marvin/ww_bonn_jax/{town}/multistart_results/{phase_cut_date}_prev{prev_phase_cut_date}_{objective}/visualizations_{cutoff_value}/ensemble_predictions_test_positive_rate.npz")


In [ ]:
import jax.numpy as jnp
import matplotlib.dates as mdates
import matplotlib.ticker as mticker

import sys 

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
from optimization import optimization_utils

# load data and base model
base_config = {
        # data selection settings
        "data_kwargs": {
            "town": town,
            "log_scale": True, # this only considers WW measurements, not case counts
        },

       "phase_cut_date": phase_cut_date, # date to split data into two phases
        "dt": 0.2,
        "T_max": 25, # dummy value

        "underreporting_model": "monotone_increasing"
}



In [ ]:
df_MSE = pd.DataFrame(columns=["Town", "Objective", "MSE Test Prev Mask", "MSE All Prev Mask"])

In [ ]:
def coverage(y, low, high):
    return ((y >= low) & (y <= high)).mean()

In [ ]:
fig, axs = plt.subplots(nrows=5, ncols=3, sharex=True, sharey="row", figsize=(13.55, 10), dpi=300, constrained_layout=True)

for j, town in enumerate(towns):
    for i, objective in enumerate(objectives):
        prev_phase_cut_date = phase_cut_date

        base_config["data_kwargs"]["town"] = town
        base_config["phase_cut_date"] = phase_cut_date
        base_config["prev_phase_cut_date"] = prev_phase_cut_date
        base_config["objective"] = objective
        hparams_path = f"/home/iru-mls/marvin/ww_bonn_jax/{town}/optuna_best_{phase_cut_date}_prev{prev_phase_cut_date}_{objective}/hparams.json"

        with open(hparams_path) as f:
            hparams = json.load(f)

        base_config.update(hparams)        
        data = optimization_utils.two_phase_integrative_model_load_data(base_config)


        quantiles_prev = {q: jnp.quantile(prev_data[town][objective]["all"], q, axis=0) for q in [0.025, 0.05, 0.25, 0.5, 0.75, 0.95, 0.975]}
        # Observations
        y_test = data["pos_tests_test"] / data["n_tests_test"] * 100
        y_all  = data["pos_tests"] / data["n_tests"] * 100

        # Masks
        mask_test = data["t_mask_prev_test"]
        mask_all  = data["t_mask_prev_all"]

        # --- 50% CI ---
        cov_50_test = coverage(
            y_test,
            quantiles_prev[0.25][mask_test],
            quantiles_prev[0.75][mask_test],
        )

        cov_50_all = coverage(
            y_all,
            quantiles_prev[0.25][mask_all],
            quantiles_prev[0.75][mask_all],
        )

        # --- 90% CI ---
        cov_90_test = coverage(
            y_test,
            quantiles_prev[0.05][mask_test],
            quantiles_prev[0.95][mask_test],
        )

        cov_90_all = coverage(
            y_all,
            quantiles_prev[0.05][mask_all],
            quantiles_prev[0.95][mask_all],
        )

        # --- 95% CI ---
        cov_95_test = coverage(
            y_test,
            quantiles_prev[0.025][mask_test],
            quantiles_prev[0.975][mask_test],
        )

        cov_95_all = coverage(
            y_all,
            quantiles_prev[0.025][mask_all],
            quantiles_prev[0.975][mask_all],
        )

        prev_median = quantiles_prev[0.5]
        df_new = pd.DataFrame({
            "Town": town,
            "Objective": objective,

            "MSE Test Prev Mask": (
                (prev_median[mask_test] - y_test) ** 2
            ).mean(),

            "MSE All Prev Mask": (
                (prev_median[mask_all] - y_all) ** 2
            ).mean(),

            # --- CI coverage ---
            "Coverage 50% Test": cov_50_test,
            "Coverage 50% All":  cov_50_all,

            "Coverage 90% Test": cov_90_test,
            "Coverage 90% All":  cov_90_all,

            "Coverage 95% Test": cov_95_test,
            "Coverage 95% All":  cov_95_all,
        }, index=[0])

        df_MSE = pd.concat([df_MSE, df_new], ignore_index=True)

        if i == 0:
            c = "#595959"
        else:
            c = "#63A066"
        axs[j, i].scatter(data["prevalence_dates_train"], data["pos_tests_train"]/data["n_tests_train"]*100, color=c, s=15, label="Training/validation\ncohort testing\ndata")
        axs[j, i].scatter(data["prevalence_dates_val"], data["pos_tests_val"]/data["n_tests_val"]*100, color=c, s=15, label=None)
        axs[j, i].scatter(data["prevalence_dates_test"], data["pos_tests_test"]/data["n_tests_test"]*100, color="#595959", s=15, label="Test")
        
        if i!=0:
            axs[j, i].axvline(pd.to_datetime(prev_phase_cut_date), color="#595959", linestyle='--', label="Phase split")
        #else:
        #    axs[j, i].axvline(pd.to_datetime("2022-10-20"), color="#595959", linestyle='--', label="Phase split")

        axs[j, i].plot(data["dates_all"], prev_median, c="#8B0000", label="Median")

        pos_rate_low  = quantiles_prev[0.25]
        pos_rate_high = quantiles_prev[0.75]
        axs[j, i].fill_between(data["dates_all"], pos_rate_low, pos_rate_high, color="#8B0000", alpha=0.45, label="50% CI")

        pos_rate_low  = quantiles_prev[0.05]
        pos_rate_high = quantiles_prev[0.95]
        axs[j, i].fill_between(data["dates_all"], pos_rate_low, pos_rate_high, color="#8B0000", alpha=0.3, label="90% CI")

        pos_rate_low  = quantiles_prev[0.025]
        pos_rate_high = quantiles_prev[0.975]
        axs[j, i].fill_between(data["dates_all"], pos_rate_low, pos_rate_high, color="#8B0000", alpha=0.15, label="95% CI")


        axs[j, i].tick_params(axis='x', rotation=45)
        if j == 0:
            axs[j, i].set_title(descriptions[i])
        
        axs[j, i].set_ylim(None, 7.5)
        # axs[i].xaxis.set_major_locator(mdates.MonthLocator(bymonth=(1, 5, 9)))
        axs[j, i].xaxis.set_major_formatter(mdates.DateFormatter('%y-%m'))

        if i==0:
            axs[j, i].set_ylabel(f"{towns[j]}\nTest positive\nrate [%]")

plt.tight_layout()
os.makedirs("sentisurv_objective_comparison", exist_ok=True)
plt.savefig(f"sentisurv_objective_comparison/all_towns.png", dpi=300, bbox_inches='tight')

In [ ]:
df_MSE.Objective.replace({'cases_and_conc': 'reported cases and wastewater data', 'prev_and_conc':'cohort testing and wastewater data', 'three_objectives': 'all three available datasets'}, inplace=True)

In [ ]:
df_MSE.rename(columns={"MSE Test Prev Mask": "MSE (test)", "MSE All Prev Mask": "MSE (all)"}, inplace=True)

In [ ]:
df_MSE

In [ ]:
import matplotlib.pyplot as plt

objectives = df_MSE["Objective"].unique()
cities = df_MSE["Town"].unique()


fig, axes = plt.subplots(
    nrows=2,
    ncols=3,
    figsize=(15.44, 5),
    dpi=300,
    sharex="col",
    gridspec_kw={"height_ratios": [1, 1], "hspace":0.1},
    constrained_layout=True,
    
)

# ======================
# Row 1: MSE
# ======================
for obj in objectives:
    df_obj = df_MSE[df_MSE["Objective"] == obj]

    axes[0, 0].plot(
        df_obj["Town"],
        df_obj["MSE (test)"],
        marker="o",
        label=obj,
    )

    axes[0, 1].plot(
        df_obj["Town"],
        df_obj["MSE (all)"],
        marker="o",
        label=obj,
    )


axes[0, 0].set_ylabel("MSE (test)")
axes[0, 1].set_ylabel("MSE (overall)")

# Legend-only panel
axes[0, 2].axis("off")
handles, labels = axes[0, 0].get_legend_handles_labels()
axes[0, 2].legend(
    handles,
    labels,
    title="Datasets used in objective",
    loc="center",
)

# ======================
# Row 2: Coverage
# ======================
for obj in objectives:
    df_obj = df_MSE[df_MSE["Objective"] == obj]

    axes[1, 0].plot(
        df_obj["Town"],
        df_obj["Coverage 50% Test"],
        marker="o",
    )

    axes[1, 1].plot(
        df_obj["Town"],
        df_obj["Coverage 90% Test"],
        marker="o",
    )

    axes[1, 2].plot(
        df_obj["Town"],
        df_obj["Coverage 95% Test"],
        marker="o",
    )


axes[1, 0].set_ylabel("Empirical coverage\n(test data, 50% CI)")
axes[1, 1].set_ylabel("\nEmpirical coverage\n(test data, 90% CI)")
axes[1, 2].set_ylabel("\nEmpirical coverage\n(test data, 95% CI)")

for ax in axes[1, :]:
    ax.set_ylim(0, 1)
    ax.tick_params(axis="x", rotation=45)


#plt.show()
plt.savefig(f"sentisurv_objective_comparison/mse_and_coverage.png", dpi=300, bbox_inches='tight')

In [ ]:
fig2, ax2 = plt.subplots(figsize=(5, 5), dpi=300)

handles, labels = [], []
i = 0
for ax in axs.ravel():
    if i == 0:
        i+=1
        continue
    h, l = ax.get_legend_handles_labels()
    for hh, ll in zip(h, l):
        if ll and ll not in labels:
            labels.append(ll)
            handles.append(hh)
#order = [0, 1, 2, 3, 4, 5, 6,]
#labels = [labels[i] for i in order]
#handles = [handles[i] for i in order]

ax2.axis('off')
fig2.legend(handles, labels, frameon=False)

fig2.savefig("sentisurv_objective_comparison/legend.png", dpi=300)